# Project 3 — Computation Graph Optimization (torch.fx)
### Conv-BN fusion + constant folding, with correctness verification

> **No GPU needed.** The default CPU runtime is correct for this project — a GPU
> would not speed it up and, for the latency measurements, would invalidate them.
> Just `Runtime` → `Run all`.

**Runtime: ~3 minutes.** The fastest project in the set.

Two hand-written FX passes. The Conv-BN folding mathematics, the pattern
matching, and the graph rewiring are all implemented here; `torch.fx` supplies
only the tracer and the graph data structure. The honest framing is *"I wrote
the optimization passes"*, not *"I wrote a compiler"*.

**Correctness is checked before any latency is reported.** If a pass changes the
model's output beyond 1e-5, the script raises and refuses to print timings — an
optimization that changes results is a bug, not an optimization.

In [ ]:
import platform, os, subprocess
print("python  ", platform.python_version())
print("arch    ", platform.machine(), "| cores", os.cpu_count())
try:
    import torch
    print("torch   ", torch.__version__, "| cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu     ", torch.cuda.get_device_name(0))
    print("quant engines", list(torch.backends.quantized.supported_engines))
except ImportError:
    pass
print(subprocess.run(["bash","-lc","grep -m1 'model name' /proc/cpuinfo || true"],
                     capture_output=True, text=True).stdout.strip())

## 1 · Write the source files

In [ ]:
!mkdir -p passes results
!touch passes/__init__.py

In [ ]:
%%writefile model_def.py
"""Target model for the FX optimization passes.

Deliberately contains BOTH optimization opportunities:

1. Four Conv2d -> BatchNorm2d -> ReLU blocks. Conv-BN is the canonical fusion
   pattern: in eval mode BatchNorm is a fixed affine map, so it can be folded
   into the preceding convolution's weights exactly.

2. A short chain of arithmetic on constant buffers (`c1 * c2 + c3`). Under FX
   these become `get_attr` nodes feeding `call_function` nodes with no
   dependency on the input placeholder -- i.e. computable at "compile" time.

The model is kept small (4 blocks) so the graph is easy to read in
`print_tabular()` output, which is itself evidence the passes did something.
"""

import torch
import torch.nn as nn


class ConvBNReLU(nn.Module):
    def __init__(self, cin: int, cout: int):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class SmallCNN(nn.Module):
    """4x (Conv-BN-ReLU) + a constant-foldable scaling tail + classifier head."""

    def __init__(self, num_classes: int = 10, width: int = 32):
        super().__init__()
        self.block1 = ConvBNReLU(3, width)
        self.block2 = ConvBNReLU(width, width * 2)
        self.pool1 = nn.MaxPool2d(2)
        self.block3 = ConvBNReLU(width * 2, width * 2)
        self.block4 = ConvBNReLU(width * 2, width * 4)
        self.pool2 = nn.MaxPool2d(2)

        # Constants stored as buffers so FX sees them as `get_attr` nodes rather
        # than folding them away at trace time (a Python literal like 2*3 would
        # already be constant-folded by the interpreter before FX ever sees it,
        # which would make the pass look like it worked when it did nothing).
        self.register_buffer("c1", torch.tensor(2.0))
        self.register_buffer("c2", torch.tensor(3.0))
        self.register_buffer("c3", torch.tensor(0.5))

        self.head = nn.Linear(width * 4, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool1(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.pool2(x)

        # --- constant subgraph: no dependency on `x`, so it is foldable ---
        k = self.c1 * self.c2      # -> 6.0
        k = k + self.c3            # -> 6.5
        k = k / self.c1            # -> 3.25
        # --- end constant subgraph ---

        x = x * k                  # depends on input: NOT foldable
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(x)


def build_model(seed: int = 42) -> nn.Module:
    """Eval-mode model with deterministic weights.

    BN fusion is only mathematically valid in eval mode, where BatchNorm uses
    its stored running statistics instead of batch statistics. The model is
    also given a few forward passes of random data first so the running stats
    are not the identity defaults -- otherwise the fusion would be trivially
    correct and would not actually test the folding math.
    """
    torch.manual_seed(seed)
    model = SmallCNN()
    model.train()
    with torch.no_grad():
        for _ in range(10):
            model(torch.randn(8, 3, 32, 32))
    model.eval()
    return model


In [ ]:
%%writefile passes/fuse_conv_bn.py
"""Conv2d + BatchNorm2d fusion pass over a torch.fx graph.

Scope, stated honestly: FX supplies the graph representation and the tracer.
The pattern matching, the folding mathematics, the construction of the
replacement module, and the graph rewiring below are hand-implemented. This is
"I wrote the optimization pass", not "I wrote a compiler".

The mathematics
---------------
In eval mode BatchNorm2d is a fixed per-channel affine transform:

    bn(y) = gamma * (y - mean) / sqrt(var + eps) + beta

and the convolution it follows is:

    y = W * x + b

Substituting, with  s = gamma / sqrt(var + eps)  (a per-output-channel scalar):

    bn(conv(x)) = s * (W * x + b - mean) + beta
                = (s * W) * x + (s * (b - mean) + beta)

So the fused convolution uses:

    W' = W * s        broadcast over the output-channel dimension
    b' = (b - mean) * s + beta

This is exact in eval mode -- not an approximation -- which is why the
correctness check downstream expects agreement to floating-point tolerance
rather than merely "close".
"""

from __future__ import annotations

import copy

import torch
import torch.fx as fx
import torch.nn as nn


def fuse_conv_bn_weights(conv: nn.Conv2d, bn: nn.BatchNorm2d) -> nn.Conv2d:
    """Return a new Conv2d equivalent to `bn(conv(x))`, with BN folded in."""
    assert not bn.training, "BN fusion is only valid in eval mode"

    fused = nn.Conv2d(
        conv.in_channels,
        conv.out_channels,
        conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        groups=conv.groups,
        bias=True,  # fusion always produces a bias, even if the conv had none
        padding_mode=conv.padding_mode,
    )

    w = conv.weight.detach().clone()
    b = (
        conv.bias.detach().clone()
        if conv.bias is not None
        else torch.zeros(conv.out_channels, dtype=w.dtype, device=w.device)
    )

    gamma = bn.weight.detach().clone()
    beta = bn.bias.detach().clone()
    mean = bn.running_mean.detach().clone()
    var = bn.running_var.detach().clone()
    eps = bn.eps

    s = gamma / torch.sqrt(var + eps)              # per-output-channel scale

    # w has shape [out, in/groups, kh, kw]; s must broadcast over dim 0 only.
    fused.weight.data = w * s.reshape(-1, 1, 1, 1)
    fused.bias.data = (b - mean) * s + beta
    return fused


def _module_of(model: nn.Module, target: str) -> nn.Module:
    mod = model
    for part in target.split("."):
        mod = getattr(mod, part)
    return mod


def _set_module(model: nn.Module, target: str, new: nn.Module) -> None:
    parts = target.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new)


def fuse_conv_bn(gm: fx.GraphModule) -> tuple[fx.GraphModule, int]:
    """Rewrite every Conv2d->BatchNorm2d pair in `gm`. Returns (module, n_fused).

    A pair is only fused when the BatchNorm consumes the convolution's output
    and *nothing else does*. If the conv output has another consumer (a skip
    connection, say), folding would silently change that other branch's input,
    so those pairs are left alone.
    """
    gm = copy.deepcopy(gm)
    modules = dict(gm.named_modules())
    n_fused = 0

    for node in list(gm.graph.nodes):
        if node.op != "call_module":
            continue
        if not isinstance(modules.get(node.target), nn.BatchNorm2d):
            continue

        (conv_node,) = node.args if len(node.args) == 1 else (node.args[0],)
        if conv_node.op != "call_module":
            continue
        conv = modules.get(conv_node.target)
        if not isinstance(conv, nn.Conv2d):
            continue

        # Guard: the conv feeds this BN and nothing else.
        if len(conv_node.users) != 1:
            continue

        bn = modules[node.target]
        fused = fuse_conv_bn_weights(conv, bn)

        _set_module(gm, conv_node.target, fused)
        modules[conv_node.target] = fused

        # Rewire: everything that read the BN output now reads the conv output.
        node.replace_all_uses_with(conv_node)
        gm.graph.erase_node(node)
        n_fused += 1

    gm.graph.lint()
    gm.recompile()  # regenerate forward() so the emitted Python matches the graph
    return gm, n_fused


In [ ]:
%%writefile passes/constant_fold.py
"""Constant-folding pass over a torch.fx graph.

Algorithm
---------
1. Mark every `placeholder` node (the model's runtime inputs) as NON-constant.
2. Propagate: a node is constant iff all of its inputs are constant.
   `get_attr` nodes -- references to parameters and buffers -- are the leaves
   that seed the constant set.
3. Evaluate each maximal constant subgraph once, eagerly, and register the
   result as a new buffer.
4. Replace uses of the subgraph's output with a `get_attr` on that buffer, then
   dead-code-eliminate the now-unused nodes.

The "maximal" part matters. Folding every constant node individually would
replace `c1*c2` with a buffer and `+c3` with another buffer and gain nothing.
Only the *last* node of a constant chain -- one whose users include at least one
non-constant node -- is materialised.

Deliberately conservative: `call_module` nodes are never folded even when their
inputs are constant, since a module could carry state (dropout, running stats)
that makes eager evaluation unsound.
"""

from __future__ import annotations

import copy

import torch
import torch.fx as fx


def _is_foldable_op(node: fx.Node) -> bool:
    # get_attr seeds the constant set; call_function / call_method propagate it.
    return node.op in ("get_attr", "call_function", "call_method")


def constant_fold(gm: fx.GraphModule) -> tuple[fx.GraphModule, dict]:
    """Fold constant subgraphs. Returns (module, stats)."""
    gm = copy.deepcopy(gm)

    # ---- step 1 + 2: mark which nodes are constant -------------------------
    is_const: dict[fx.Node, bool] = {}
    for node in gm.graph.nodes:
        if node.op == "placeholder":
            is_const[node] = False
        elif node.op == "output":
            is_const[node] = False
        elif node.op == "get_attr":
            is_const[node] = True
        elif _is_foldable_op(node):
            inputs = list(node.all_input_nodes)
            is_const[node] = bool(inputs) and all(is_const.get(i, False) for i in inputs)
        else:
            is_const[node] = False  # call_module: conservatively not folded

    const_nodes = [n for n, c in is_const.items() if c and n.op != "get_attr"]

    # ---- step 3: find the frontier -- constant nodes with a non-constant user
    frontier = [
        n for n in const_nodes
        if any(not is_const.get(u, False) for u in n.users)
    ]

    if not frontier:
        return gm, {"folded_subgraphs": 0, "nodes_removed": 0, "values": {}}

    # Evaluate the whole graph's constant part once via an FX interpreter run.
    # Any real input works -- constant nodes ignore it by construction.
    env: dict[fx.Node, object] = {}

    def _resolve(a):
        return fx.node.map_arg(a, lambda n: env[n])

    for node in gm.graph.nodes:
        if not is_const.get(node, False):
            continue
        if node.op == "get_attr":
            env[node] = _fetch_attr(gm, node.target)
        elif node.op == "call_function":
            env[node] = node.target(*_resolve(node.args), **_resolve(node.kwargs))
        elif node.op == "call_method":
            self_obj, *rest = _resolve(node.args)
            env[node] = getattr(self_obj, node.target)(*rest, **_resolve(node.kwargs))

    # ---- step 4: materialise and rewire ------------------------------------
    values = {}
    for i, node in enumerate(frontier):
        value = env[node]
        if not isinstance(value, torch.Tensor):
            value = torch.tensor(value)
        buf_name = f"_folded_const_{i}"
        gm.register_buffer(buf_name, value.detach().clone())
        values[buf_name] = value.detach().clone().flatten().tolist()

        with gm.graph.inserting_before(node):
            new_node = gm.graph.get_attr(buf_name)
        node.replace_all_uses_with(new_node)

    before = len(list(gm.graph.nodes))
    gm.graph.eliminate_dead_code()
    gm.graph.lint()
    gm.recompile()
    after = len(list(gm.graph.nodes))

    return gm, {
        "folded_subgraphs": len(frontier),
        "nodes_removed": before - after,
        "values": values,
    }


def _fetch_attr(mod: torch.nn.Module, target: str):
    obj = mod
    for part in target.split("."):
        obj = getattr(obj, part)
    return obj


In [ ]:
%%writefile benchmark.py
"""Correctness verification + latency benchmark for the FX passes.

Order of operations matters: correctness is checked BEFORE any timing is
reported. A pass that changes the model's output is a bug, and a latency number
from a broken pass is worse than no number at all.

All timing is CPU wall-clock via time.perf_counter(), warmup discarded, every
raw sample written to disk.
"""

import argparse
import json
import os
import platform
import subprocess
import time

import numpy as np
import torch
import torch.fx as fx

from model_def import build_model
from passes.constant_fold import constant_fold
from passes.fuse_conv_bn import fuse_conv_bn

TORCH_THREADS = 4
TOL = 1e-5


def hardware() -> dict:
    cpu = "unknown"
    try:
        if platform.system() == "Darwin":
            cpu = subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        else:
            out = subprocess.check_output(
                ["bash", "-lc", "grep -m1 'model name' /proc/cpuinfo"], text=True)
            cpu = out.split(":", 1)[1].strip()
    except Exception:
        pass
    return {
        "cpu": cpu, "arch": platform.machine(), "logical_cores": os.cpu_count(),
        "torch_threads": TORCH_THREADS, "torch": torch.__version__,
        "python": platform.python_version(),
    }


@torch.no_grad()
def latency(model, x, runs=300, warmup=50):
    for _ in range(warmup):
        model(x)
    s = []
    for _ in range(runs):
        t0 = time.perf_counter()
        model(x)
        s.append((time.perf_counter() - t0) * 1000)
    return s


def summary(s):
    a = np.array(s)
    return {
        "n": int(a.size), "mean_ms": float(a.mean()), "std_ms": float(a.std(ddof=1)),
        "p50_ms": float(np.percentile(a, 50)), "p95_ms": float(np.percentile(a, 95)),
        "p99_ms": float(np.percentile(a, 99)), "min_ms": float(a.min()),
    }


def node_counts(gm: fx.GraphModule) -> dict:
    c = {}
    for n in gm.graph.nodes:
        c[n.op] = c.get(n.op, 0) + 1
    c["total"] = sum(v for k, v in c.items() if k != "total")
    return c


def tabular(gm: fx.GraphModule) -> str:
    import io
    from contextlib import redirect_stdout

    buf = io.StringIO()
    with redirect_stdout(buf):
        gm.graph.print_tabular()
    return buf.getvalue()


@torch.no_grad()
def max_abs_diff(a_model, b_model, inputs) -> float:
    worst = 0.0
    for x in inputs:
        d = (a_model(x) - b_model(x)).abs().max().item()
        worst = max(worst, d)
    return worst


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--runs", type=int, default=300)
    p.add_argument("--warmup", type=int, default=50)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--out", default="results/fx_results.json")
    args = p.parse_args()

    torch.manual_seed(42)
    torch.set_num_threads(TORCH_THREADS)
    os.makedirs("results", exist_ok=True)

    model = build_model()
    gm = fx.symbolic_trace(model)

    variants = {"baseline (traced)": gm}
    fused, n_fused = fuse_conv_bn(gm)
    variants["conv-bn fused"] = fused
    folded, fold_stats = constant_fold(gm)
    variants["const folded"] = folded
    both, _ = constant_fold(fused)
    variants["fused + folded"] = both

    print(f"[fx] conv-bn pairs fused: {n_fused}")
    print(f"[fx] constant subgraphs folded: {fold_stats['folded_subgraphs']} "
          f"-> values {fold_stats['values']}")

    # ---- correctness FIRST -------------------------------------------------
    check_inputs = [torch.randn(args.batch_size, 3, 32, 32) for _ in range(10)]
    print("\n[check] verifying every optimized graph against the original")
    correctness = {}
    all_ok = True
    for name, m in variants.items():
        m.eval()
        d = max_abs_diff(model, m, check_inputs)
        ok = bool(d < TOL)
        all_ok &= ok
        correctness[name] = {"max_abs_diff": d, "allclose_tol": TOL, "passed": ok}
        print(f"[check] {name:20s} max|diff| = {d:.3e}   {'PASS' if ok else 'FAIL'}")

    if not all_ok:
        raise SystemExit("[check] a pass changed the model output -- refusing to report latency")

    # ---- structure ---------------------------------------------------------
    counts = {name: node_counts(m) for name, m in variants.items()}
    print("\n[graph] node counts")
    for name, c in counts.items():
        print(f"[graph] {name:20s} total={c['total']:3d}  "
              + "  ".join(f"{k}={v}" for k, v in sorted(c.items()) if k != "total"))

    graphs = {name: tabular(m) for name, m in variants.items()}
    with open("results/graphs_before_after.txt", "w") as f:
        for name, t in graphs.items():
            f.write(f"{'=' * 78}\n{name}\n{'=' * 78}\n{t}\n\n")
    print("[graph] wrote results/graphs_before_after.txt")

    # ---- latency -----------------------------------------------------------
    x = torch.randn(args.batch_size, 3, 32, 32)
    print(f"\n[bench] batch={args.batch_size}, {args.runs} runs, {args.warmup} warmup")
    raw, lat = {}, {}
    for name, m in variants.items():
        s = latency(m, x, args.runs, args.warmup)
        raw[name] = s
        lat[name] = summary(s)
        print(f"[bench] {name:20s} p50={lat[name]['p50_ms']:7.3f}ms  "
              f"p95={lat[name]['p95_ms']:7.3f}ms  mean={lat[name]['mean_ms']:7.3f}ms")

    base = lat["baseline (traced)"]["p50_ms"]
    for name, s in lat.items():
        s["speedup_vs_baseline"] = base / s["p50_ms"]
        s["latency_reduction_pct"] = 100 * (1 - s["p50_ms"] / base)

    print("\n[bench] p50 latency reduction vs traced baseline")
    for name, s in lat.items():
        print(f"[bench] {name:20s} {s['latency_reduction_pct']:+6.2f}%  "
              f"({s['speedup_vs_baseline']:.3f}x)")

    payload = {
        "hardware": hardware(),
        "protocol": {
            "runs": args.runs, "warmup": args.warmup, "batch_size": args.batch_size,
            "timed_region": "model(x) forward only",
            "correctness_tolerance": TOL,
            "correctness_inputs": "10 random batches, max abs diff vs eager model",
        },
        "passes": {"conv_bn_pairs_fused": n_fused, "constant_fold": fold_stats},
        "correctness": correctness,
        "node_counts": counts,
        "latency": lat,
    }
    with open(args.out, "w") as f:
        json.dump(payload, f, indent=2)
    with open("results/latency_raw.json", "w") as f:
        json.dump(raw, f)
    print(f"\n[bench] wrote {args.out}")


if __name__ == "__main__":
    main()


## 2 · Run the passes, verify correctness, benchmark

In [ ]:
!python benchmark.py --runs 300 --warmup 50 --batch-size 16

## 3 · The graphs, before and after

This is the structural evidence that the passes actually transformed the graph
rather than merely producing the same numbers.

In [ ]:
txt = open("results/graphs_before_after.txt").read()
print(txt[:6000])

In [ ]:
import json
d = json.load(open("results/fx_results.json"))
print("node counts:", json.dumps(d["node_counts"], indent=2))
print("\ncorrectness:", json.dumps(d["correctness"], indent=2))

## 4 · Batch-size sweep

Fusion wins come from removing kernel launches and intermediate memory traffic,
so the benefit varies with batch size. Worth reporting across a few points
rather than picking the flattering one.

In [ ]:
import json, subprocess, pandas as pd
rows = []
for bs in [1, 8, 32, 64]:
    subprocess.run(["python","benchmark.py","--runs","200","--warmup","40",
                    "--batch-size",str(bs),"--out",f"results/fx_bs{bs}.json"], check=True)
    d = json.load(open(f"results/fx_bs{bs}.json"))
    r = {"batch": bs}
    for k, v in d["latency"].items():
        r[k] = round(v["p50_ms"], 3)
        r[k + " %"] = round(v["latency_reduction_pct"], 2)
    rows.append(r)
df = pd.DataFrame(rows); df.to_csv("results/batch_sweep.csv", index=False); df

## 5 · Download

In [ ]:
import os
folders = [f for f in "results".split() if os.path.isdir(f)]
!zip -qr project03_results.zip {" ".join(folders)}
print("zipped:", folders)
from google.colab import files
files.download("project03_results.zip")